# Exploración de datos

___

## El Dataset

Este conjunto de datos contiene estudios de resonancia magnética (RM) de rodilla anotados con doce hallazgos comunes: lesiones de ligamentos y meniscos, osteoartritis en tres compartimentos, derrame articular, sinovitis, quiste de Baker, contusión ósea y fractura. Cada estudio comprende una colección de secuencias de RM individuales de una sola sesión de exploración, formateadas como series DICOM.

Los estudios provienen de una variada mezcla internacional de centros de diagnóstico por imágenes y abarcan una amplia gama de escáneres, protocolos y poblaciones. Solo un pequeño subconjunto de estudios de entrenamiento cuenta con etiquetas por afección. También se proporciona el texto original del informe radiológico.

### Estructura del directorio

    data 
    ├── sample_submission.csv 
    ├── test.csv 
    ├── test_series 
    │ㅤㅤ└── <StudyInstanceUID> 
    │ㅤㅤㅤㅤㅤ└── <SeriesInstanceUID> 
    │ㅤㅤㅤㅤㅤㅤㅤㅤ└── <SOPInstanceUID>.dcm
    ├── test_series.csv 
    ├── train.csv 
    ├── train_series 
    │ㅤㅤ└── <StudyInstanceUID> 
    │ㅤㅤㅤㅤㅤ└── <SeriesInstanceUID> 
    │ㅤㅤㅤㅤㅤㅤㅤㅤ└── <SOPInstanceUID>.dcm 
    └── train_series.csv 

### Especificación de archivos

##### **data**
Carpeta principal que contiene los archivos.


##### **sample_submission.csv**
Es un archivo csv como ejemplo de cómo deben ser las predicciones. Son 13 columnas, primero es el ID de una instancia en específico, seguido son 12 columnas donde cada una contiene la probabilidad (0.0 - 1.0) de pertenecer a cada una de las clases. En este caso para ejemplicifación todas fueron establecidas en 0.5 de probabilidad.


##### **test.csv**
Contiene 3 IDs de estudio del set de datos de testeo público. Durante la evaluación este set de datos es reemplazado con los datos de testing reales, los cuales conforman 1300 estudios.


##### **test_series/** 
Archivos DICOM organizados de la siguiente forma: 

        test_series 
    ㅤㅤㅤ└── <StudyInstanceUID> 
    ㅤㅤㅤㅤㅤㅤ└── <SeriesInstanceUID> 
    ㅤㅤㅤㅤㅤㅤㅤㅤ└── <SOPInstanceUID>.dcm


##### **test_series.csv**
Este csv contiene 5 columnas distribuídas de la siguiente forma:
1. StudyInstanceUID. Es el estudio al que pertenecen las series.
2. SeriesInstanceUID. Es el ID de una serie que coincide con la carpeta train_series/\<StudyInstanceUID>.
3. Fluid_Sensitive. Indica "1" si la secuencia se enfoca en señales de líquido (T2, PD, STIR y similares); indica "0" en caso contrario.
4. Fat_Suppression. Indica "1" si la secuencia aplica supresión de grasa; indica "0" en caso contrario.
5. Anatomical_Plane. Indica el plano de la imagen, estos pueden ser "Sagittal", "Coronal" o "Axial".

##### **train.csv**
Cada fila es un estudio para entrenamiento y se divide en las siguientes 14 columnas:
1. StudyInstanceUID. El ID de los estudios.
2. Report. Es el reporte de la radiología, es un reporte por estudio, no uno por serie. Varía el lenguaje en el que está escrito el reporte.
3. ACL. lesión del ligamento cruzado anterior (0/1).
4. MCL. lesión del ligamento colateral medial (0/1).
5. Median Meniscus. rotura del menisco medial (0/1).
6. Lateral Meniscus. rotura del menisco lateral (0/1).
7. Medial OA. osteoartritis del compartimento tibiofemoral medial (0/1).
8. Lateral OA. osteoartritis del compartimento tibiofemoral lateral (0/1).
9. PF OA. osteoartritis patelofemoral (0/1).
10. Effusion: derrame articular / exceso de líquido (0/1).
11. Synovitis: inflamación de la membrana sinovial (0/1).
12. Baker's. quiste de Baker (0/1).
13. Contusion. contusión ósea / hematoma óseo (0/1).
14. Fracture. fractura (0/1).

##### **train_series/**
Igual a la estructura de [test_series](#test_series)

##### **train_series.csv**
Igual a la estructura de [test_series.csv](#test_seriescsv)

### Los archivos DICOM

#### ¿Qué es un archivo DICOM?
DICOM (Digital Imaging and Communications in Medicine) es el formato estándar para guardar y transmitir imágenes médicas. Lo usan los resonadores, tomógrafos y equipos de rayos X de casi cualquier hospital. Cada archivo .dcm tiene dos partes:

1. La imagen: una matriz de píxeles, casi siempre en escala de grises y con más profundidad que una foto normal (12 o 16 bits en lugar de 8).
2. Los metadatos (el header): cientos de campos con información del estudio, por ejemplo:
identificadores (StudyInstanceUID, SeriesInstanceUID, SOPInstanceUID), que son los mismos números largos de tus CSV y nombres de carpetas;
geometría: tamaño del píxel en mm (PixelSpacing), grosor del corte (SliceThickness), posición y orientación del corte en el espacio (ImagePositionPatient, ImageOrientationPatient);
parámetros de adquisición: tiempos de la secuencia de resonancia (RepetitionTime, EchoTime), el fabricante del equipo, la descripción de la secuencia;
el orden del corte (InstanceNumber).

#### Estructura de nuestros archivos DICOM
**\<StudyInstanceUID>/** \
Identifica el estudio, una visita del paciente al resonador para revisar una rodilla. Es la unidad de predicción: cada renglón de train.csv y de submission.csv es un estudio con sus 12 etiquetas.

**\<SeriesInstanceUID>/** \
Identifica una serie dentro del estudio, es decir, una secuencia adquirida con un plano y un contraste determinados (por ejemplo, sagital con supresión de grasa).

**\<SOPInstanceUID>.dcm** \
Identifica una instancia, una sola rebanada (slice) dentro de la serie. Cada archivo .dcm lleva este identificador como nombre. SOP viene de Service-Object Pair, un término del estándar que aquí basta con leer como "imagen individual".

**En resumen** \
Cada carpeta **\<StudyInstanceUID>/** contiene un estudio, o sea, una visita de un paciente. Cabe aclarar que un paciente puede tener más de un estudio (no se especifica en Kaggle si esto pasa). \
Cada carpeta **\<SeriesInstanceUID>/** es una serie que contiene múltiples archivos *.dcm* que pertenecen a una sola secuencia y cada secuencia puede ser sagital, coronal o axial, con y sin supresión de grasa, sensible o no a fluídos. \
Cada **\<SOPInstanceUID>.dcm** es una rebanada 2D de una serie.